In [80]:
import sys
import numpy as np
import torch 
# still do preprocessing in scipy
import scipy.sparse as sp
import anndata as ad 
from importlib import reload
from scipy.sparse import coo_matrix

import h5py
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

# make UMAP using average assign_post across seeds and color points by cell_type 
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

from tqdm import tqdm
import pandas as pd

# import factor model from beta-dirichlet-factor
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/beta-dirichlet-factor')
import factor_model
reload(factor_model)

sns.set(style='white', context='notebook', rc={'figure.figsize':(6,6)}, font_scale=1.5)

# Append this directory to sys.path
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/')
import load_cluster_data as llc 
reload(llc)

# Append also simulation directory
sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/")
import simulate_counts as sim 
reload (sim)

sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/visualization/")
import vis as vis

# import factor model from beta-dirichlet-factor
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/evaluations')
import masking_BBFactor as mask 
reload(mask) 

import cost_correlation_assign
from cost_correlation_assign import *

reload(cost_correlation_assign)
from differential_splicing import * 

2.3.0+cu121
12.1
Using device: cpu


In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

float_type = { 
        "device": device, 
        "dtype": torch.float,
    }

input_conc = torch.tensor(np.inf, **float_type)
print(input_conc)

Using device: cpu
tensor(inf)


In [3]:
float_type

{'device': device(type='cpu'), 'dtype': torch.float32}

### Load real brain single cell data from Tabula Senis

In [4]:
adata = ad.read_h5ad('/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leafletall_ages_brain_intron_clusters_adata.h5ad')
adata.obs["cell_id_index"] = adata.obs.index
print(f"Original Cluster_Counts nnz: {adata.layers['Cluster_Counts'].count_nonzero()}")
print(f"Original Junction_Counts nnz: {adata.layers['Junction_Counts'].count_nonzero()}")

#cell_type_column = "cell_ontology_class"
#filter_clusters = True

cell_type_column = None
filter_clusters = False # in this case maybe better to first assign random clustesr outside of simulaiton code 
K_use = 3

# Load Anndata object
adata_input = adata.copy()  # Make a copy to avoid modifying the original data
proportion_negative = 0.5

# If cell type column is present in adata.obs, then set K to the number of unique cell types
if cell_type_column in adata.obs.columns:
    K = adata.obs[cell_type_column].nunique()
    
    # Print the unique cell types and their counts
    unique_cell_types = adata.obs[cell_type_column].value_counts()
    print(f"Unique cell types in '{cell_type_column}':")
    print(unique_cell_types)

else:
    K = K_use

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Original Cluster_Counts nnz: 29565846
Original Junction_Counts nnz: 13523365


In [5]:
29565846-13523365

16042481

### Remove lowly observed ATSEs!!

In [6]:
if filter_clusters:
    
    # Extract the cluster counts matrix (C x J) and cell types
    cluster_counts = adata_input.layers["Cluster_Counts"]
    cell_types = adata_input.obs[cell_type_column].values

    print(adata_input.layers["Cluster_Counts"].shape, adata_input.layers["Junction_Counts"].shape)

    # Get unique cell types and clusters
    unique_cell_types = np.unique(cell_types)
    unique_clusters = adata_input.var_names # assuming var_names corresponds to junction clusters

    # Initialize a DataFrame to store the counts
    expression_counts = pd.DataFrame(0, index=unique_cell_types, columns=unique_clusters)

    # Calculate the number of cells expressing each intron cluster for each cell type
    for cell_type in tqdm(unique_cell_types):
        cells_in_type = (cell_types == cell_type)
        counts_in_type = cluster_counts.toarray()[cells_in_type, :].sum(axis=0) # number of cells in cell type found for each cluster pair 
        expression_counts.loc[cell_type] = counts_in_type  # .A converts sparse matrix to dense array

    # Calculate the number of cell types where each cluster has non-zero (and > 5) expression
    non_zero_counts = (expression_counts >= 5).sum(axis=0)

    # Calculate the threshold for filtering (20% of total cell types)
    threshold = len(expression_counts) * 0.2
    print(threshold) 

    # Filter clusters that are expressed in more than 25% of cell types
    filtered_clusters = non_zero_counts[non_zero_counts > threshold].index

    # Subset the original DataFrame to keep only the filtered clusters
    filtered_expression_counts = expression_counts[filtered_clusters]

    # Juncs to keep 
    juncs_keep = list(filtered_expression_counts)

    # Subset Anndata vars to only these adata_input.var_names along with adata layers 
    adata_filtered = adata_input[:, juncs_keep].copy()

    # need to ensure indices of junctions/clustesr are still concordant across var info and counts matrices 
    adata_filtered.var['junction_id_index'] = np.arange(adata_filtered.n_vars)
    adata_filtered.var_names = adata_filtered.var['junction_id_index'].astype(str)
    print(adata_filtered.layers["Cluster_Counts"].shape, adata_filtered.layers["Junction_Counts"].shape)

else:
    adata_filtered = adata_input.copy()

### Simulate data!

In [7]:
print(f"Original Cluster_Counts nnz: {adata_filtered.layers['Cluster_Counts'].count_nonzero()}")
print(f"Original Junction_Counts nnz: {adata_filtered.layers['Junction_Counts'].count_nonzero()}")

Original Cluster_Counts nnz: 29565846
Original Junction_Counts nnz: 13523365


In [11]:
adata_input = sim.simulate_and_prepare_data(adata_filtered, K, float_type, proportion_negative, cell_type_column)

# Index clean up 
adata_input.obs.drop(columns=['cell_id_index'], inplace=True)
adata_input.obs.reset_index(inplace=True)

Cluster_Counts nnz: 10020363
Junction_Counts nnz: 5217550
The number of unique junctions included in the simulation data is: 17703
The number of unique clusters included in the simulation data is: 5901


100%|██████████| 5901/5901 [00:15<00:00, 381.62it/s]
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Cluster_Counts nnz: 10020363
Junction_Counts nnz: 5217550
The proportion of negative ASEs to set is: 0.5
The number of cell types is: 3
The number of cells is: 12657
The number of junctions is: 17703
Number of negative labels (0): 2950
Number of positive labels (1): 2951


100%|██████████| 5901/5901 [00:29<00:00, 202.41it/s]


Assertion passed: 'junction_id_index' matches the index in 'adata_input.var'.
Done simulating PSI!
Done simulating junction counts!
True label counts:
 true_label
negative    9414
positive    8289
Name: count, dtype: int64
Sample label counts:
 sample_label
positive    8853
negative    8850
Name: count, dtype: int64
Cluster_Counts nnz: 10020363
Junction_Counts nnz: 9048454
Data successfully simulated and prepared!


In [12]:
adata_input

AnnData object with n_obs × n_vars = 12657 × 17703
    obs: 'cell_id_index', 'cell_id', 'age', 'batch', 'cell_type', 'mouse.id', 'sex', 'subtissue', 'tissue', 'louvain', 'leiden'
    var: 'Cluster', 'junction_id', 'gene_id', 'chr', 'start', 'end', 'junction_id_index', 'index', 0, 1, 2, 'sample_label', 'difference', 'true_label'
    layers: 'Cluster_Counts', 'Junction_Counts', 'junc_ratio'

In [81]:
# use adata_filtered to mask data 

# 1. Generate the mask using the AnnData object (adata_input)
mask_gen, seed = mask.generate_mask(adata_input, layer_key="Cluster_Counts", mask_percentage=0.1)


Total masked entries: 1002036.0


In [82]:
reload(mask)
reload(llc)

Using device: cpu


<module 'load_cluster_data' from '/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/load_cluster_data.py'>

In [85]:
print(f"Original Cluster_Counts nnz: {adata_input.layers['Cluster_Counts'].count_nonzero()}")
print(f"Original Junction_Counts nnz: {adata_input.layers['Junction_Counts'].count_nonzero()}")


Original Cluster_Counts nnz: 10020363
Original Junction_Counts nnz: 9048454


In [84]:
# 2. Apply the mask and create a new AnnData object:
reload(llc)
reload(mask)
masked_adata, my_data = mask.apply_mask_to_anndata(adata_input, mask_gen, cluster_layer="Cluster_Counts", junction_layer="Junction_Counts")

Using device: cpu
Masked_Cluster_Counts nnz: 9018327
Masked_Junction_Counts nnz: 8143549


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Unique cell indices: 12657
8143549


In [86]:
# Remake my_data object for model training 
my_data

IndexCountTensor(ycount_lookup=tensor(crow_indices=tensor([      0,     458,     921,  ..., 8143044,
                            8143124, 8143549]),
       col_indices=tensor([    0,     1,   147,  ..., 17622, 17623, 17624]),
       values=tensor([  1.,   3., 214.,  ...,  96.,   6.,  93.]),
       size=(12657, 17703), nnz=8143549, layout=torch.sparse_csr), tcount_lookup=tensor(crow_indices=tensor([      0,     458,     921,  ..., 8143044,
                            8143124, 8143549]),
       col_indices=tensor([    0,     1,   147,  ..., 17622, 17623, 17624]),
       values=tensor([  6.,   6., 544.,  ..., 204., 204., 204.]),
       size=(12657, 17703), nnz=8143549, layout=torch.sparse_csr), ycount_lookup_T=tensor(crow_indices=tensor([      0,     414,     844,  ..., 8143537,
                            8143544, 8143549]),
       col_indices=tensor([   0,   13,   24,  ..., 5499, 5570, 6910]),
       values=tensor([1., 1., 2.,  ..., 2., 2., 1.]), size=(17703, 12657),
       nnz=8143549,

In [87]:
if device == torch.device('cuda'):
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#### TO FIX BUG: SOMETIMES I GET --> 193 assert torch.all(assign >= 0), "Assign has negative values!"
    #194 # Assert that values in pi sum up to 1 
    #195 assert torch.allclose(pi.sum(), torch.tensor(1.0)), f"pi does not sum to 1: {pi.sum()}"
    # AssertionError: Assign has negative values!

In [88]:
reload(mask)

Using device: cpu


<module 'masking_BBFactor' from '/gpfs/commons/home/kisaev/Leaflet-private/src/evaluations/masking_BBFactor.py'>

In [89]:
reload(factor_model)
print(K)

# make input conc infinity first 
input_conc = torch.tensor(np.inf, **float_type)
plot_bb=False
# Convert the CSR tensor to COO format
full_y_tensor = my_data.ycount_lookup.to_sparse_coo()
full_total_counts_tensor = my_data.tcount_lookup.to_sparse_coo()
all_results, variable_sizes = factor_model.main(full_y_tensor, full_total_counts_tensor, num_initializations=2, use_global_prior=False, K=K, lr=0.1, input_conc_prior=input_conc, loss_plot=False, num_epochs=100, save_to_file = False)

# figure out which index in all_results has the lowest final loss
losses = [result["losses"][-1] for result in all_results]
# get number of epochs for each initialization
num_epochs = [len(result["losses"]) for result in all_results]

# make a quick plot of the losses
plt.plot(losses)
plt.xlabel("Initialization")
plt.ylabel("Final Loss")
# make each loss value with a dot 
plt.scatter(range(len(losses)), losses)

# also mark the number of epochs
for i in range(len(losses)):
    plt.text(i, losses[i], str(num_epochs[i]), size="small")
# mark the lowest loss red
plt.scatter(np.argmin(losses), np.min(losses), color="red")

# get the index of initialization with the lowest loss
best_init = np.argmin(losses)
print("The best initialization was: ", best_init)

# extract latent variables for just first seed used 
latent_vars = all_results[best_init]['summary_stats'] 

pi = latent_vars["pi"]["mean"] # overall contribution of each factor to cell population, one value per k
dir_conc = latent_vars["dir_conc"]["mean"] # one scaling value 
assign_post = latent_vars["assign"]["mean"]
psis = latent_vars["psi"]["mean"] # psi is the probability of a junction being used in a cluster
a = latent_vars["a"]["mean"] 
b = latent_vars["b"]["mean"] 


Using a fixed probability of success across all trials (infinite concentration parameter) with a binomial distribution.
Not using priors on a and b, running simpler non-hierarchical version!


2.3.0+cu121
12.1
3
Random seeds: [9724, 265]
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Initialization 1 with seed 9724
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Define the guide using AutoDiagonalNormal based on the model structure.
Fit the model
cpu
Training in progress for 100 epochs!
Epoch 0, Elbo loss: 409737917.5508526


KeyboardInterrupt: 